# Phase 10 — Export Deployment Artifacts for FastAPI

Notebook này **không benchmark lại Phase 9** và **không cần chạy Phase 9 trong cùng runtime**.

Nó mount Google Drive rồi đọc trực tiếp các đường dẫn/artifact đang dùng trong Phase 9 để tạo một bộ deployment sạch cho FastAPI, Docker, Linux GPU server và streaming SSE.

## Artifact đầu ra

```text
vn_history_deployment/
├── model/
│   └── qwen2_5_3b_vnhistory_stage12_merged/
├── retrieval/
│   ├── faiss/
│   └── bm25s_index/
├── corpus/
│   └── vn_history_rag_chunks_enriched.jsonl
├── config/
│   └── inference_config.json
├── evaluation/
├── manifest.json
└── EXPORT_SUCCESS.txt
```

## Streaming design

Config deployment được chuẩn bị cho **SSE validated streaming**:

1. retrieval + reranking;
2. Qwen generation;
3. source/year guards + critic + optional repair;
4. chỉ answer cuối đã được chấp nhận mới stream ra client.

Như vậy demo vẫn có hiệu ứng streaming nhưng không phá cơ chế grounded-generation của Phase 9.

**GPU khuyến nghị:** L4 24 GB hoặc A100. T4 16 GB có thể chạy nhưng nên theo dõi VRAM khi merge model.

## 1. Cài dependencies

Chỉ cài các thư viện cần để merge LoRA, đọc FAISS/BM25 và export model.

In [1]:
# Cell 1 — Install export dependencies

%pip -q install -U \
  "transformers>=4.51,<5" \
  "peft>=0.13,<1" \
  "accelerate>=1.0,<2" \
  "safetensors>=0.4" \
  "faiss-cpu>=1.8" \
  "torchao==0.17.0" \
  "bm25s>=0.2.14"

print("✅ Export dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 119.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 105.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
✅ Export dependencies installed.


## 2. Mount Google Drive + cấu hình

Các path dưới đây được lấy từ Phase 9 hiện tại.

In [2]:
# Cell 2 — Mount Drive + Phase 9 source paths + Phase 10 output paths

from google.colab import drive
drive.mount("/content/drive")

import os
import re
import gc
import json
import shutil
import zipfile
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List

import torch
import faiss
import bm25s

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

PIPELINE_VERSION = "phase9_v2_tooluse_grounded_direct"
PHASE10_EXPORT_VERSION = "phase10_fastapi_export_v1"

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

DRIVE_ROOT = Path("/content/drive/MyDrive")
BACKUP_DIR = DRIVE_ROOT / "vn_history_model_backups"
RAG_ROOT = BACKUP_DIR / "rag_corpus_vn_history"
PHASE8_METADATA_DIR = RAG_ROOT / "metadata"
PHASE9_DIR = RAG_ROOT / "phase9_hybrid_rag"

ENRICHED_CANDIDATES = [
    PHASE8_METADATA_DIR / "vn_history_rag_chunks_enriched.jsonl",
    RAG_ROOT / "processed" / "vn_history_rag_chunks_enriched.jsonl",
]

RAW_CORPUS_PATH = RAG_ROOT / "processed" / "vn_history_rag_chunks.jsonl"
METADATA_PATH = PHASE8_METADATA_DIR / "vn_history_rag_chunk_metadata.jsonl"

PHASE1_ADAPTER_CANDIDATES = [
    BACKUP_DIR / "qwen_vnhistory_phase1_best_adapter",
    BACKUP_DIR / "qwen_vnhistory_phase1_best_adapter.zip",
]

PHASE2_ADAPTER_CANDIDATES = [
    BACKUP_DIR / "qwen_vnhistory_phase6_rag_best_adapter",
    BACKUP_DIR / "qwen_vnhistory_phase6_rag_best_adapter.zip",
    BACKUP_DIR / "qwen2_5_3b_vnhistory_phase6_rag_qlora_best_by_generation_metric",
    BACKUP_DIR / "qwen2_5_3b_vnhistory_phase6_rag_qlora_best_by_generation_metric.zip",
]

EMBEDDING_MODEL_ID = "intfloat/multilingual-e5-base"
RERANKER_MODEL_ID = "BAAI/bge-reranker-v2-m3"

safe_embed = re.sub(r"[^A-Za-z0-9._-]+", "_", EMBEDDING_MODEL_ID)

PHASE9_FAISS_DIR = PHASE9_DIR / f"faiss_{safe_embed}"
PHASE9_FAISS_INDEX_PATH = PHASE9_FAISS_DIR / "chunks.index"
PHASE9_FAISS_MANIFEST_PATH = PHASE9_FAISS_DIR / "manifest.json"

PHASE9_BM25_DIR = PHASE9_DIR / "bm25s_index"
PHASE9_BM25_MANIFEST_PATH = PHASE9_BM25_DIR / "phase9_manifest.json"

# Phase 10 deployment root
DEPLOY_ROOT = BACKUP_DIR / "vn_history_deployment"

MODEL_EXPORT_DIR = DEPLOY_ROOT / "model" / "qwen2_5_3b_vnhistory_stage12_merged"
FAISS_EXPORT_DIR = DEPLOY_ROOT / "retrieval" / "faiss"
BM25_EXPORT_DIR = DEPLOY_ROOT / "retrieval" / "bm25s_index"
CORPUS_EXPORT_DIR = DEPLOY_ROOT / "corpus"
CONFIG_EXPORT_DIR = DEPLOY_ROOT / "config"
EVAL_EXPORT_DIR = DEPLOY_ROOT / "evaluation"

for p in [
    MODEL_EXPORT_DIR,
    FAISS_EXPORT_DIR,
    BM25_EXPORT_DIR,
    CORPUS_EXPORT_DIR,
    CONFIG_EXPORT_DIR,
    EVAL_EXPORT_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

DEPLOY_CORPUS_PATH = CORPUS_EXPORT_DIR / "vn_history_rag_chunks_enriched.jsonl"
FAISS_EXPORT_PATH = FAISS_EXPORT_DIR / "chunks.index"
FAISS_EXPORT_MANIFEST_PATH = FAISS_EXPORT_DIR / "manifest.json"
CONFIG_PATH = CONFIG_EXPORT_DIR / "inference_config.json"
MANIFEST_PATH = DEPLOY_ROOT / "manifest.json"
SUCCESS_PATH = DEPLOY_ROOT / "EXPORT_SUCCESS.txt"

EXPECTED_CHUNKS = 58_603

FORCE_REEXPORT_MODEL = False
FORCE_RECOPY_RETRIEVAL = False
FORCE_RECOPY_CORPUS = False
HASH_LARGE_MODEL_FILES = False

print("Phase 9 source :", PHASE9_DIR)
print("Phase 10 output:", DEPLOY_ROOT)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {GPU_VRAM_GB:.1f} GB")
else:
    GPU_VRAM_GB = 0.0
    print("⚠️ Không có GPU. Merge Qwen 3B trên CPU sẽ rất chậm.")

Mounted at /content/drive


Phase 9 source : /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag
Phase 10 output: /content/drive/MyDrive/vn_history_model_backups/vn_history_deployment
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB


## 3. Utilities

Resolve corpus/adapters, checksum và copy helper. Adapter dạng folder hoặc ZIP đều được hỗ trợ như Phase 9.

In [3]:
# Cell 3 — Utilities

def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def resolve_enriched_corpus() -> Path:
    for p in ENRICHED_CANDIDATES:
        if p.exists() and p.stat().st_size > 0:
            return p
    raise FileNotFoundError(
        "Không tìm thấy enriched corpus đã dùng trong Phase 9.\n"
        + "\n".join(str(x) for x in ENRICHED_CANDIDATES)
    )


def resolve_adapter_dir(name: str, candidates: List[Path]) -> Path:
    extract_root = Path("/content/phase10_adapters") / name
    extract_root.parent.mkdir(parents=True, exist_ok=True)

    for raw in candidates:
        p = Path(raw)

        if p.is_dir() and (p / "adapter_config.json").exists():
            return p

        if p.is_file() and p.suffix.lower() == ".zip":
            if (extract_root / "adapter_config.json").exists():
                return extract_root

            print(f"Extracting {name} adapter:", p)

            if extract_root.exists():
                shutil.rmtree(extract_root)

            extract_root.mkdir(parents=True, exist_ok=True)

            with zipfile.ZipFile(p, "r") as z:
                z.extractall(extract_root)

            matches = list(extract_root.rglob("adapter_config.json"))
            if not matches:
                raise FileNotFoundError(f"Không thấy adapter_config.json trong {p}")

            return matches[0].parent

    matches = list(BACKUP_DIR.rglob("adapter_config.json"))
    for m in matches:
        if name.lower() in str(m).lower():
            return m.parent

    raise FileNotFoundError(f"Không tìm thấy adapter {name}. Candidates: {candidates}")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            block = f.read(chunk_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()


def copy_file_if_needed(src: Path, dst: Path, force: bool = False):
    src = Path(src)
    dst = Path(dst)

    if not src.exists():
        raise FileNotFoundError(src)

    dst.parent.mkdir(parents=True, exist_ok=True)

    same_size = (
        dst.exists()
        and dst.is_file()
        and dst.stat().st_size == src.stat().st_size
    )

    if force or not same_size:
        shutil.copy2(src, dst)
        print("Copied:", src, "→", dst)
    else:
        print("Already exported:", dst)


def copy_dir_fresh(src: Path, dst: Path, force: bool = False):
    src = Path(src)
    dst = Path(dst)

    if not src.is_dir():
        raise FileNotFoundError(src)

    if dst.exists() and force:
        shutil.rmtree(dst)

    if not dst.exists():
        shutil.copytree(src, dst)
        print("Copied directory:", src, "→", dst)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Synced directory:", src, "→", dst)


def model_export_valid(path: Path) -> bool:
    path = Path(path)

    if not (path / "config.json").exists():
        return False

    if not (path / "tokenizer_config.json").exists():
        return False

    weights = list(path.glob("*.safetensors"))
    if not weights:
        return False

    return all(x.stat().st_size > 0 for x in weights)


def count_jsonl_rows(path: Path) -> int:
    n = 0
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                n += 1
    return n


print("✅ Utilities ready.")

✅ Utilities ready.


## 4. Preflight Phase 9 artifacts

Xác nhận corpus, adapters, FAISS và BM25 đồng bộ trước khi merge/export.

In [4]:
# Cell 4 — Preflight Phase 9 artifacts

ENRICHED_CORPUS_PATH = resolve_enriched_corpus()

PHASE1_ADAPTER_DIR = resolve_adapter_dir(
    "phase1",
    PHASE1_ADAPTER_CANDIDATES,
)

PHASE2_ADAPTER_DIR = resolve_adapter_dir(
    "phase2",
    PHASE2_ADAPTER_CANDIDATES,
)

assert ENRICHED_CORPUS_PATH.exists()
assert (PHASE1_ADAPTER_DIR / "adapter_config.json").exists()
assert (PHASE2_ADAPTER_DIR / "adapter_config.json").exists()
assert PHASE9_FAISS_INDEX_PATH.exists()
assert PHASE9_FAISS_MANIFEST_PATH.exists()
assert PHASE9_BM25_DIR.is_dir()
assert PHASE9_BM25_MANIFEST_PATH.exists()

corpus_count = count_jsonl_rows(ENRICHED_CORPUS_PATH)

faiss_manifest = json.loads(
    PHASE9_FAISS_MANIFEST_PATH.read_text(encoding="utf-8")
)

bm25_manifest = json.loads(
    PHASE9_BM25_MANIFEST_PATH.read_text(encoding="utf-8")
)

faiss_check = faiss.read_index(str(PHASE9_FAISS_INDEX_PATH))

print("Enriched corpus :", ENRICHED_CORPUS_PATH)
print("Phase1 adapter  :", PHASE1_ADAPTER_DIR)
print("Phase2 adapter  :", PHASE2_ADAPTER_DIR)
print("FAISS index     :", PHASE9_FAISS_INDEX_PATH)
print("BM25 directory  :", PHASE9_BM25_DIR)

print("\n===== PRE-FLIGHT =====")
print("Corpus rows          :", corpus_count)
print("FAISS ntotal         :", faiss_check.ntotal)
print("FAISS manifest count :", faiss_manifest.get("count"))
print("BM25 manifest count  :", bm25_manifest.get("count"))
print("Corpus signature     :", faiss_manifest.get("corpus_signature"))

assert corpus_count == EXPECTED_CHUNKS
assert faiss_check.ntotal == corpus_count
assert int(faiss_manifest.get("count", -1)) == corpus_count
assert int(bm25_manifest.get("count", -1)) == corpus_count

if (
    faiss_manifest.get("corpus_signature")
    and bm25_manifest.get("corpus_signature")
):
    assert (
        faiss_manifest["corpus_signature"]
        == bm25_manifest["corpus_signature"]
    )

del faiss_check

print("\n✅ Phase 9 artifacts đồng bộ.")

Enriched corpus : /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/metadata/vn_history_rag_chunks_enriched.jsonl
Phase1 adapter  : /content/drive/MyDrive/vn_history_model_backups/qwen_vnhistory_phase1_best_adapter
Phase2 adapter  : /content/drive/MyDrive/vn_history_model_backups/qwen_vnhistory_phase6_rag_best_adapter
FAISS index     : /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/faiss_intfloat_multilingual-e5-base/chunks.index
BM25 directory  : /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/bm25s_index

===== PRE-FLIGHT =====
Corpus rows          : 58603
FAISS ntotal         : 58603
FAISS manifest count : 58603
BM25 manifest count  : 58603
Corpus signature     : ef60222b683077c6b72c

✅ Phase 9 artifacts đồng bộ.


## 5. Merge Phase 1 + Phase 2 → final deployment model

Quy trình giữ đúng Phase 9:

```text
Qwen2.5-3B-Instruct
→ Phase 1 LoRA
→ merge
→ Phase 2 LoRA
→ merge
→ save safetensors
```

Nếu model đã export hợp lệ và `FORCE_REEXPORT_MODEL=False`, cell sẽ skip.

In [5]:
# Cell 5 — Merge and export Stage1+Stage2 final model

if model_export_valid(MODEL_EXPORT_DIR) and not FORCE_REEXPORT_MODEL:
    print("✅ Final merged model đã tồn tại → skip merge.")
    print(MODEL_EXPORT_DIR)

else:
    if not torch.cuda.is_available():
        print("⚠️ Không có CUDA. Merge trên CPU sẽ rất chậm.")

    dtype = (
        torch.bfloat16
        if torch.cuda.is_available()
        and torch.cuda.get_device_capability(0)[0] >= 8
        else torch.float16
    )

    device_map = {"": 0} if torch.cuda.is_available() else None

    print("Loading tokenizer:", MODEL_ID)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        use_fast=True,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenizer.padding_side = "left"

    print("Loading base model:", MODEL_ID)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=dtype,
        device_map=device_map,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )

    model.config.pad_token_id = tokenizer.pad_token_id

    print("Loading Phase1 adapter → merge...")

    model = PeftModel.from_pretrained(
        model,
        str(PHASE1_ADAPTER_DIR),
        is_trainable=False,
    ).merge_and_unload()

    cleanup_cuda()

    print("Loading Phase2 adapter on Phase1-merged model → merge...")

    model = PeftModel.from_pretrained(
        model,
        str(PHASE2_ADAPTER_DIR),
        is_trainable=False,
    ).merge_and_unload()

    cleanup_cuda()

    model.eval()
    model.config.use_cache = True
    model.config.pad_token_id = tokenizer.pad_token_id

    try:
        model.generation_config.do_sample = False
        model.generation_config.temperature = None
        model.generation_config.top_p = None
        model.generation_config.top_k = None
    except Exception:
        pass

    if MODEL_EXPORT_DIR.exists() and FORCE_REEXPORT_MODEL:
        shutil.rmtree(MODEL_EXPORT_DIR)

    MODEL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    print("Saving merged model to Drive...")

    model.save_pretrained(
        MODEL_EXPORT_DIR,
        safe_serialization=True,
        max_shard_size="4GB",
    )

    tokenizer.save_pretrained(MODEL_EXPORT_DIR)

    if getattr(model, "generation_config", None) is not None:
        model.generation_config.save_pretrained(MODEL_EXPORT_DIR)

    print("✅ Final model saved:", MODEL_EXPORT_DIR)

    del model
    del tokenizer
    cleanup_cuda()

assert model_export_valid(MODEL_EXPORT_DIR)

print("\nModel shards:")
for p in sorted(MODEL_EXPORT_DIR.glob("*.safetensors")):
    print(f" - {p.name}: {p.stat().st_size / (1024**3):.2f} GB")

Loading tokenizer: Qwen/Qwen2.5-3B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading Phase1 adapter → merge...
Loading Phase2 adapter on Phase1-merged model → merge...
Saving merged model to Drive...
✅ Final model saved: /content/drive/MyDrive/vn_history_model_backups/vn_history_deployment/model/qwen2_5_3b_vnhistory_stage12_merged

Model shards:
 - model-00001-of-00002.safetensors: 3.70 GB
 - model-00002-of-00002.safetensors: 2.05 GB


## 6. Export corpus + FAISS + BM25

Copy đúng retrieval artifacts đã benchmark ở Phase 9; không rebuild embeddings/index ở đây.

In [6]:
# Cell 6 — Copy corpus and retrieval artifacts

copy_file_if_needed(
    ENRICHED_CORPUS_PATH,
    DEPLOY_CORPUS_PATH,
    force=FORCE_RECOPY_CORPUS,
)

copy_file_if_needed(
    PHASE9_FAISS_INDEX_PATH,
    FAISS_EXPORT_PATH,
    force=FORCE_RECOPY_RETRIEVAL,
)

copy_file_if_needed(
    PHASE9_FAISS_MANIFEST_PATH,
    FAISS_EXPORT_MANIFEST_PATH,
    force=FORCE_RECOPY_RETRIEVAL,
)

copy_dir_fresh(
    PHASE9_BM25_DIR,
    BM25_EXPORT_DIR,
    force=FORCE_RECOPY_RETRIEVAL,
)

print("\n✅ Corpus + retrieval export complete.")

Copied: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/metadata/vn_history_rag_chunks_enriched.jsonl → /content/drive/MyDrive/vn_history_model_backups/vn_history_deployment/corpus/vn_history_rag_chunks_enriched.jsonl
Copied: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/faiss_intfloat_multilingual-e5-base/chunks.index → /content/drive/MyDrive/vn_history_model_backups/vn_history_deployment/retrieval/faiss/chunks.index
Copied: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/faiss_intfloat_multilingual-e5-base/manifest.json → /content/drive/MyDrive/vn_history_model_backups/vn_history_deployment/retrieval/faiss/manifest.json
Synced directory: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/bm25s_index → /content/drive/MyDrive/vn_history_model_backups/vn_history_deployment/retrieval/bm25s_index

✅ Corpus + retrieval export complete.


## 7. Export inference config

FastAPI sau này sẽ đọc file JSON này thay vì hard-code các tham số Phase 9.

In [7]:
# Cell 7 — Write FastAPI/deployment inference config

DEFAULT_SYSTEM = (
    "Bạn là trợ lý AI chuyên về lịch sử Việt Nam. "
    "Trả lời trực tiếp đúng trọng tâm, rõ ràng và chính xác. "
    'Không mở đầu bằng các câu rập khuôn như "Theo tài liệu". '
    "Nếu không đủ cơ sở để khẳng định, nói rõ mức độ không chắc chắn."
)

SAFE_OOD_ANSWER = (
    "Câu hỏi này nằm ngoài phạm vi hệ thống lịch sử Việt Nam, "
    "nên tôi không trả lời bằng corpus hiện tại."
)

SAFE_INSUFFICIENT_ANSWER = (
    "Không đủ bằng chứng trong các tài liệu truy xuất "
    "để trả lời chắc chắn câu hỏi này."
)

inference_config = {
    "phase9_pipeline_version": PIPELINE_VERSION,
    "phase10_export_version": PHASE10_EXPORT_VERSION,

    "model": {
        "base_model_id": MODEL_ID,
        "deployment_model_path": "model/qwen2_5_3b_vnhistory_stage12_merged",
        "merged_stage1_stage2": True,
        "dtype_preference": "float16_or_bfloat16",
    },

    "retrieval": {
        "embedding_model_id": EMBEDDING_MODEL_ID,
        "reranker_model_id": RERANKER_MODEL_ID,
        "dense_fetch_k": 80,
        "bm25_fetch_k": 80,
        "rrf_k": 60,
        "rrf_top_k": 20,
        "final_context_k": 6,
        "rerank_batch_size": 32,
        "max_query_variants": 3,
        "query_expansion_weight": 0.82,
        "max_chunks_per_title": 2,
        "enable_context_diversity": True,
        "metadata_max_bonus": 0.18,
        "intent_facet_bonus": 0.025,
    },

    "prompt": {
        "max_input_tokens": 3600,
        "max_new_tokens": 300,
        "max_chars_per_chunk": 1800,
        "min_chars_per_chunk": 550,
        "default_system": DEFAULT_SYSTEM,
    },

    "generation": {
        "temperature": 0.0,
        "top_p": 1.0,
        "repetition_penalty": 1.05,
        "use_cache": True,
    },

    "guards": {
        "strict_source_required": True,
        "strict_unsupported_year_guard": True,
        "enable_completeness_rewrite": True,
        "max_rewrite_attempts": 1,
        "safe_ood_answer": SAFE_OOD_ANSWER,
        "safe_insufficient_answer": SAFE_INSUFFICIENT_ANSWER,
    },

    "ood": {
        "anchor_margin": 0.02,
        "secondary_margin": -0.06,
        "secondary_min_dense": 0.28,
    },

    "streaming": {
        "enabled": True,
        "transport": "sse",
        "mode": "validated_streaming",
        "description": (
            "Retrieval, generation, guards and optional repair run first; "
            "only the final accepted answer is streamed to the client."
        ),
        "planned_events": [
            "retrieval_started",
            "reranking",
            "generating",
            "validating",
            "answer_delta",
            "sources",
            "done",
            "error",
        ],
    },

    "deployment": {
        "api_version": "v1",
        "framework": "FastAPI",
        "recommended_gpu": "NVIDIA L4 24GB",
        "faiss_device": "cpu",
        "bm25_device": "cpu",
    },
}

write_json(CONFIG_PATH, inference_config)

print("✅ Inference config exported:")
print(CONFIG_PATH)
print(json.dumps(inference_config, ensure_ascii=False, indent=2)[:5000])

✅ Inference config exported:
/content/drive/MyDrive/vn_history_model_backups/vn_history_deployment/config/inference_config.json
{
  "phase9_pipeline_version": "phase9_v2_tooluse_grounded_direct",
  "phase10_export_version": "phase10_fastapi_export_v1",
  "model": {
    "base_model_id": "Qwen/Qwen2.5-3B-Instruct",
    "deployment_model_path": "model/qwen2_5_3b_vnhistory_stage12_merged",
    "merged_stage1_stage2": true,
    "dtype_preference": "float16_or_bfloat16"
  },
  "retrieval": {
    "embedding_model_id": "intfloat/multilingual-e5-base",
    "reranker_model_id": "BAAI/bge-reranker-v2-m3",
    "dense_fetch_k": 80,
    "bm25_fetch_k": 80,
    "rrf_k": 60,
    "rrf_top_k": 20,
    "final_context_k": 6,
    "rerank_batch_size": 32,
    "max_query_variants": 3,
    "query_expansion_weight": 0.82,
    "max_chunks_per_title": 2,
    "enable_context_diversity": true,
    "metadata_max_bonus": 0.18,
    "intent_facet_bonus": 0.025
  },
  "prompt": {
    "max_input_tokens": 3600,
    "max_

## 8. Copy Phase 9 benchmark evidence

Không bắt buộc để inference chạy, nhưng rất hữu ích để giữ traceability giữa model deployment và kết quả evaluation.

In [8]:
# Cell 8 — Copy evaluation outputs if available

evaluation_candidates = [
    PHASE9_DIR / "benchmark_100.jsonl",
    PHASE9_DIR / "benchmark_results_v3_unique_batched.jsonl",
    PHASE9_DIR / "benchmark_summary_v3_unique_batched.csv",
    PHASE9_DIR / "benchmark_results_v2_tooluse.jsonl",
    PHASE9_DIR / "benchmark_summary_v2_tooluse.csv",
]

copied_eval = []

for src in evaluation_candidates:
    if src.exists() and src.is_file():
        dst = EVAL_EXPORT_DIR / src.name
        copy_file_if_needed(src, dst, force=False)
        copied_eval.append(dst.name)

print("Evaluation files exported:", copied_eval)

Copied: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/benchmark_results_v3_unique_batched.jsonl → /content/drive/MyDrive/vn_history_model_backups/vn_history_deployment/evaluation/benchmark_results_v3_unique_batched.jsonl
Copied: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/phase9_hybrid_rag/benchmark_summary_v3_unique_batched.csv → /content/drive/MyDrive/vn_history_model_backups/vn_history_deployment/evaluation/benchmark_summary_v3_unique_batched.csv
Evaluation files exported: ['benchmark_results_v3_unique_batched.jsonl', 'benchmark_summary_v3_unique_batched.csv']


## 9. Build deployment manifest

Manifest ghi source artifacts, version, counts, checksums và danh sách model shards.

In [9]:
# Cell 9 — Build deployment manifest

model_files = []

for p in sorted(MODEL_EXPORT_DIR.rglob("*")):
    if not p.is_file():
        continue

    rec = {
        "path": str(p.relative_to(DEPLOY_ROOT)),
        "size_bytes": p.stat().st_size,
    }

    if HASH_LARGE_MODEL_FILES or p.stat().st_size < 50 * 1024 * 1024:
        rec["sha256"] = sha256_file(p)

    model_files.append(rec)

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "phase9_pipeline_version": PIPELINE_VERSION,
    "phase10_export_version": PHASE10_EXPORT_VERSION,

    "source": {
        "phase9_directory": str(PHASE9_DIR),
        "phase1_adapter": str(PHASE1_ADAPTER_DIR),
        "phase2_adapter": str(PHASE2_ADAPTER_DIR),
        "enriched_corpus": str(ENRICHED_CORPUS_PATH),
    },

    "corpus": {
        "path": str(DEPLOY_CORPUS_PATH.relative_to(DEPLOY_ROOT)),
        "count": corpus_count,
        "size_bytes": DEPLOY_CORPUS_PATH.stat().st_size,
        "sha256": sha256_file(DEPLOY_CORPUS_PATH),
    },

    "faiss": {
        "path": str(FAISS_EXPORT_PATH.relative_to(DEPLOY_ROOT)),
        "count": int(faiss_manifest["count"]),
        "embedding_model": faiss_manifest.get("embedding_model"),
        "dim": faiss_manifest.get("dim"),
        "corpus_signature": faiss_manifest.get("corpus_signature"),
        "size_bytes": FAISS_EXPORT_PATH.stat().st_size,
        "sha256": sha256_file(FAISS_EXPORT_PATH),
    },

    "bm25": {
        "path": str(BM25_EXPORT_DIR.relative_to(DEPLOY_ROOT)),
        "count": int(bm25_manifest["count"]),
        "corpus_signature": bm25_manifest.get("corpus_signature"),
    },

    "model": {
        "base_model_id": MODEL_ID,
        "merged_stage1_stage2": True,
        "files": model_files,
    },

    "config": {
        "path": str(CONFIG_PATH.relative_to(DEPLOY_ROOT)),
        "sha256": sha256_file(CONFIG_PATH),
    },

    "evaluation_files": copied_eval,
    "streaming": inference_config["streaming"],
}

write_json(MANIFEST_PATH, manifest)

print("✅ Manifest exported:")
print(MANIFEST_PATH)

✅ Manifest exported:
/content/drive/MyDrive/vn_history_model_backups/vn_history_deployment/manifest.json


## 10. Final sanity check

Chỉ khi cell này pass toàn bộ thì bộ artifact mới được xem là sẵn sàng cho FastAPI.

In [10]:
# Cell 10 — Final deployment artifact validation

checks = {}

checks["model_valid"] = model_export_valid(MODEL_EXPORT_DIR)

checks["corpus_exists"] = DEPLOY_CORPUS_PATH.exists()
deploy_corpus_count = (
    count_jsonl_rows(DEPLOY_CORPUS_PATH)
    if DEPLOY_CORPUS_PATH.exists()
    else -1
)
checks["corpus_count"] = deploy_corpus_count == EXPECTED_CHUNKS

checks["faiss_exists"] = FAISS_EXPORT_PATH.exists()

if FAISS_EXPORT_PATH.exists():
    deploy_faiss = faiss.read_index(str(FAISS_EXPORT_PATH))
    deploy_faiss_count = int(deploy_faiss.ntotal)
    del deploy_faiss
else:
    deploy_faiss_count = -1

checks["faiss_count"] = deploy_faiss_count == EXPECTED_CHUNKS

checks["bm25_dir"] = BM25_EXPORT_DIR.is_dir()
checks["bm25_manifest"] = (
    BM25_EXPORT_DIR / "phase9_manifest.json"
).exists()

if checks["bm25_manifest"]:
    deploy_bm25_manifest = json.loads(
        (BM25_EXPORT_DIR / "phase9_manifest.json").read_text(
            encoding="utf-8"
        )
    )
    deploy_bm25_count = int(
        deploy_bm25_manifest.get("count", -1)
    )
else:
    deploy_bm25_count = -1

checks["bm25_count"] = deploy_bm25_count == EXPECTED_CHUNKS
checks["config"] = CONFIG_PATH.exists()
checks["manifest"] = MANIFEST_PATH.exists()

print("\n" + "=" * 72)
print("PHASE 10 — DEPLOYMENT ARTIFACT CHECK")
print("=" * 72)

for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)

print("\nCounts")
print("Corpus :", deploy_corpus_count)
print("FAISS  :", deploy_faiss_count)
print("BM25   :", deploy_bm25_count)

all_ok = all(checks.values())

if not all_ok:
    failed = [name for name, ok in checks.items() if not ok]
    raise RuntimeError(
        "Phase 10 export chưa hoàn chỉnh. Failed: "
        + ", ".join(failed)
    )

success_lines = [
    "PHASE 10 EXPORT SUCCESS",
    "",
    f"Created UTC: {datetime.now(timezone.utc).isoformat()}",
    f"Phase 9 pipeline: {PIPELINE_VERSION}",
    f"Phase 10 export: {PHASE10_EXPORT_VERSION}",
    "",
    f"Corpus chunks: {deploy_corpus_count}",
    f"FAISS vectors: {deploy_faiss_count}",
    f"BM25 corpus count: {deploy_bm25_count}",
    "",
    "Model:",
    str(MODEL_EXPORT_DIR),
    "",
    "Deployment root:",
    str(DEPLOY_ROOT),
    "",
    "Streaming:",
    "SSE validated streaming enabled in inference_config.json",
]

SUCCESS_PATH.write_text(
    "\n".join(success_lines),
    encoding="utf-8",
)

print("\n✅ DEPLOYMENT ARTIFACTS READY")
print("Root  :", DEPLOY_ROOT)
print("Marker:", SUCCESS_PATH)


PHASE 10 — DEPLOYMENT ARTIFACT CHECK
✅ model_valid
✅ corpus_exists
✅ corpus_count
✅ faiss_exists
✅ faiss_count
✅ bm25_dir
✅ bm25_manifest
✅ bm25_count
✅ config
✅ manifest

Counts
Corpus : 58603
FAISS  : 58603
BM25   : 58603

✅ DEPLOYMENT ARTIFACTS READY
Root  : /content/drive/MyDrive/vn_history_model_backups/vn_history_deployment
Marker: /content/drive/MyDrive/vn_history_model_backups/vn_history_deployment/EXPORT_SUCCESS.txt


# Sau Phase 10

Không cần chạy lại Phase 9.

Bước tiếp theo là **FastAPI application** đọc trực tiếp bộ artifact đã export:

```text
GET  /health
GET  /ready
POST /api/v1/retrieve
POST /api/v1/chat
POST /api/v1/chat/stream
```

`/api/v1/chat/stream` sẽ dùng **Server-Sent Events (SSE)** với `validated_streaming` làm mặc định.